In [105]:
import torch.nn as nn
import numpy as np
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import time
from tokenizers import Tokenizer

In [106]:
VOCAB_SIZE  = 30000 # Number of tokens in the vocabulary
SEQ_LEN     = 8      # Context Length
D_MODEL     = 8      # Input hidden dim

# ── MLA-specific dims ─────────────────────────────────────────────────────────
N_HEADS      = 2     # Number of attention heads
D_HEAD_NOPE  = 4     # NoPE (content) dim per head
D_HEAD_ROPE  = 2     # RoPE (positional) dim per head
D_HEAD       = D_HEAD_NOPE + D_HEAD_ROPE   # Effective full head dim for attention
D_C_KV       = 4    # KV latent compression dim 
D_C_Q        = 4    # Q  latent compression dim 

N_MTP      = 3    # Predict N_MTP  future tokens
MTP_LAMBDA = 0.3  # Auxiliary loss weight: Loss = Loss(next_token) + MTP_LAMBDA(Loss(mtp))

D_FF        = 16 # Expansion dim of the feedforward network in the transformer block
DROPOUT     = 0.1 # Randomly zeroes DROPOUT*100% activations during training.
BATCH_SIZE  = 64 # Each training step processes BATCH_SIZE sequences in parallel
LR          = 3e-4 # Optimizer step size
EPOCHS      = 1 # Number of times to loop through the entire training dataset

TOKENIZER_PATH = Path.cwd() / "models" / "tokenizer.json"
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
print(f"input shape per sequence: ({SEQ_LEN}, {D_MODEL})")
print(f"W_DKV shape: ({D_MODEL}, {D_C_KV})")

device: cuda
input shape per sequence: (8, 8)
W_DKV shape: (8, 4)


## Creating dataset from the tokenized sequences

In [107]:
TRAIN_BIN = Path.cwd() / "data" / "train.bin"

token_ids = np.fromfile(TRAIN_BIN, dtype=np.uint16).astype(np.int64)
print(f"total tokens: {len(token_ids):,}")

class TokenDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.ids) - self.seq_len

    def __getitem__(self, idx):
        x = self.ids[idx : idx + self.seq_len]
        y = self.ids[idx + 1 : idx + self.seq_len + 1]
        return x, y

dataset    = TokenDataset(token_ids, SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f"batches per epoch: {len(dataloader)}")

total tokens: 136,633
batches per epoch: 2134


## Positional Encoding
$$z_{rotated} = (x_0 \cos \theta - x_1 \sin \theta) + i (x_0 \sin \theta + x_1 \cos \theta)$$

In [108]:
# Decoupled RoPE utilities
# Applied ONLY to the Q_rope / K_rope pathways, not to the NoPE content.

def precompute_freqs_cis(d_rope: int, max_seq_len: int, theta: float = 10000.0):
    """
    Returns complex unit vectors for RoPE rotation.
    d_rope must be even.
    Output shape: (max_seq_len, d_rope // 2)  — complex64
    """
    assert d_rope % 2 == 0
    # Inverse frequencies: θ_i = 1 / 10000^(2i / d_rope)
    inv_freq = 1.0 / (theta ** (torch.arange(0, d_rope, 2).float() / d_rope))
    t        = torch.arange(max_seq_len, dtype=torch.float32) # Tells position of the token in the sequence
    # Outer product -> Angles matrix  (max_seq_len, d_rope//2),  
    angles   = torch.outer(t, inv_freq) # Value at row m and column i is the exact rotation angle in radians needed for that specific token position and feature pair.
    # e^{i·angle} = cos + i·sin  — the rotation in complex form
    freqs_cis = torch.polar(torch.ones_like(angles), angles) 
    return freqs_cis   # (T, d_rope//2)  complex64


def apply_rope(x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
    """
    Rotate x using pre-computed complex frequencies.
    x         : (B, n_heads, T, d_rope)   — real
    freqs_cis : (T, d_rope // 2)          — complex
    Returns   : (B, n_heads, T, d_rope)   — real, rotated
    """
    B, H, T, d = x.shape
    freqs_cis = freqs_cis.to(x.device)
    # pair up consecutive dims as Re/Im of a complex number
    x_c = torch.view_as_complex(x.float().reshape(B, H, T, d // 2, 2))  # (B,H,T,d/2) complex, (xo,x1) -> (xo + i·x1)
    # broadcast freqs over batch and heads
    f   = freqs_cis[:T].unsqueeze(0).unsqueeze(0)    # (1, 1, T, d/2) complex
    # multiply = rotate in 2D plane
    out = torch.view_as_real(x_c * f).reshape(B, H, T, d)
    return out.to(x.dtype)

# Pre-compute once for the full context length
freqs_cis = precompute_freqs_cis(D_HEAD_ROPE, SEQ_LEN).to(DEVICE)
print(f"freqs_cis shape: {freqs_cis.shape}  (T={SEQ_LEN}, d_rope/2={D_HEAD_ROPE//2})  dtype={freqs_cis.dtype}")

freqs_cis shape: torch.Size([8, 1])  (T=8, d_rope/2=1)  dtype=torch.complex64


# RoPE Demo

In [109]:
cis = precompute_freqs_cis(4, SEQ_LEN)
x = torch.ones(1, 1, SEQ_LEN, 4)  # (B,H,T,d_rope)
out = apply_rope(x, cis)
print("Input tensor:")
print(x)
print("\nPre-computed freqs_cis:")
print(cis)
print("\nRotated tensor:")
print(out)

Input tensor:
tensor([[[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [1., 1., 1., 1.]]]])

Pre-computed freqs_cis:
tensor([[ 1.0000+0.0000j,  1.0000+0.0000j],
        [ 0.5403+0.8415j,  0.9999+0.0100j],
        [-0.4161+0.9093j,  0.9998+0.0200j],
        [-0.9900+0.1411j,  0.9996+0.0300j],
        [-0.6536-0.7568j,  0.9992+0.0400j],
        [ 0.2837-0.9589j,  0.9988+0.0500j],
        [ 0.9602-0.2794j,  0.9982+0.0600j],
        [ 0.7539+0.6570j,  0.9976+0.0699j]])

Rotated tensor:
tensor([[[[ 1.0000,  1.0000,  1.0000,  1.0000],
          [-0.3012,  1.3818,  0.9900,  1.0099],
          [-1.3254,  0.4932,  0.9798,  1.0198],
          [-1.1311, -0.8489,  0.9696,  1.0295],
          [ 0.1032, -1.4104,  0.9592,  1.0392],
          [ 1.2426, -0.6753,  0.9488,  1.0487],
          [ 1.2396,  0.6808,  0.9382,  1.0582],
          [ 0.0969,  1.4109,

## Multi-Head Latent Attention (MLA)

```
h_t  (B, T, 8)
 │
 ├─ W_DKV (8×4) ──→ c_kv (B,T,4)  ← cached at inference
 │    ├─ W_UK (4×8) ──→ K_nope (B,H,T,4)  ┐ per-head content
 │    └─ W_UV (4×8) ──→ V      (B,H,T,4)  ┘
 │
 ├─ W_DQ  (8×4) ──→ c_q  (B,T,4)
 │    ├─ W_UQ (4×8) ──→ Q_nope (B,H,T,4)          per-head content
 │    └─ W_QR (4×4) ──→ Q_rope (B,H,T,2) ──RoPE── per-head positional query
 │
 └─ W_KR  (8×2) ──→ K_rope (B,1,T,2) ──RoPE── SHARED positional key (broadcast)
                                                  ↓
             Q = [Q_nope ; Q_rope]  (B,H,T,6)   per-head
             K = [K_nope ; K_rope]  (B,H,T,6)   K_rope broadcast from (B,1,T,2)
             attn = softmax(Q·Kᵀ / √6) · V
```

`K_rope` is **head-shared**: `W_KR` outputs `d_head_rope` (not `n_heads × d_head_rope`).
The same positional key vector is appended to every head's key, saving `(n_heads - 1) × d_head_rope` in the KV cache per token.

In [110]:
class MultiHeadLatentAttention(nn.Module):
    """
    DeepSeek-V2 style MLA with decoupled RoPE.
    """
    def __init__(self, d_model: int = D_MODEL, n_heads: int = N_HEADS, d_head_nope: int = D_HEAD_NOPE, d_head_rope: int = D_HEAD_ROPE,
                       d_c_kv: int = D_C_KV, d_c_q: int = D_C_Q, dropout: float = DROPOUT):
        super().__init__()
        self.n_heads     = n_heads
        self.d_head_nope = d_head_nope
        self.d_head_rope = d_head_rope
        self.d_head      = d_head_nope + d_head_rope
        self.scale       = self.d_head ** -0.5

        # ── KV pathway ───────────────────────────────────────────────────────
        self.W_DKV = nn.Linear(d_model, d_c_kv, bias=False)
        self.W_UK  = nn.Linear(d_c_kv,  n_heads*d_head_nope, bias=False)
        self.W_UV  = nn.Linear(d_c_kv,  n_heads*d_head_nope, bias=False)

        # ── Q pathway ────────────────────────────────────────────────────────
        self.W_DQ  = nn.Linear(d_model, d_c_q, bias=False)
        self.W_UQ  = nn.Linear(d_c_q,   n_heads*d_head_nope, bias=False)

        # ── Decoupled RoPE pathway ────────────────────────────────────────────
        self.W_QR = nn.Linear(d_c_q, n_heads*d_head_rope, bias=False)
        self.W_KR  = nn.Linear(d_model, d_head_rope, bias=False)  # HEAD-SHARED

        # ── Output ───────────────────────────────────────────────────────────
        self.W_O   = nn.Linear(n_heads*d_head_nope, d_model, bias=False)
        self.drop  = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        """
        h         : (B, T, d_model)
        freqs_cis : (T, d_head_rope//2)  complex
        returns   : (B, T, d_model)
        """
        B, T, _ = h.shape

        # ── Step 1 · Compress ────────────────────────────────────────────────
        c_kv = self.W_DKV(h)   # (B, T, d_c_kv=4)
        c_q  = self.W_DQ(h)    # (B, T, d_c_q=4)

        # ── Step 2 · NoPE up-projection (per-head) ───────────────────────────
        def to_heads(x, d_h):
            return x.view(B, T, self.n_heads, d_h).transpose(1, 2)

        K_nope = to_heads(self.W_UK(c_kv), self.d_head_nope)  # (B, H, T, 4)
        V      = to_heads(self.W_UV(c_kv), self.d_head_nope)  # (B, H, T, 4)
        Q_nope = to_heads(self.W_UQ(c_q),  self.d_head_nope)  # (B, H, T, 4)

        # ── Step 3 · Decoupled RoPE pathway ──────────────────────────────────
        # Q_rope: per-head
        Q_rope = to_heads(self.W_QR(c_q), self.d_head_rope)
        Q_rope = apply_rope(Q_rope, freqs_cis)

        # K_rope: HEAD-SHARED — one vector per token, same for all heads
        K_rope = self.W_KR(h)                                    # (B, T, d_head_rope=2)
        K_rope = K_rope.unsqueeze(1)                             # (B, 1, T, 2)
        K_rope = apply_rope(K_rope, freqs_cis)                   # (B, 1, T, 2)  rotated
        K_rope = K_rope.expand(-1, self.n_heads, -1, -1)        # (B, H, T, 2)  broadcast

        # ── Step 4 · Concatenate NoPE + RoPE ─────────────────────────────────
        Q = torch.cat([Q_nope, Q_rope], dim=-1)  # (B, H, T, 6)
        K = torch.cat([K_nope, K_rope], dim=-1)  # (B, H, T, 6)

        # ── Step 5 · Causal attention ─────────────────────────────────────────
        attn = (Q @ K.transpose(-2, -1)) * self.scale
        mask = torch.tril(torch.ones(T, T, device=h.device, dtype=torch.bool))
        attn = attn.masked_fill(~mask, float("-inf"))
        attn = F.softmax(attn, dim=-1)
        attn = self.drop(attn)

        out = attn @ V                                            # (B, H, T, 4)
        out = out.transpose(1, 2).contiguous()
        out = out.view(B, T, self.n_heads * self.d_head_nope)
        return self.W_O(out)                                      # (B, T, d_model)

    def kv_cache_size_bytes(self, seq_len: int) -> dict:
        fp16 = 2
        # MLA caches: c_kv (shared) + K_rope (head-shared, NOT per-head)
        mla_total = (seq_len * D_C_KV + seq_len * self.d_head_rope) * fp16
        mha_total = seq_len * self.n_heads * self.d_head * fp16 * 2  # K + V
        return {
            "MHA  K+V per token": f"{self.n_heads}×{self.d_head} + {self.n_heads}×{self.d_head} = {2*self.n_heads*self.d_head} values",
            "MLA  c_kv + K_rope": f"{D_C_KV} + {self.d_head_rope} = {D_C_KV + self.d_head_rope} values  (K_rope shared, not ×{self.n_heads})",
            "bytes MHA": mha_total,
            "bytes MLA": mla_total,
            "compression_ratio": round(mha_total / mla_total, 2),
        }


mla = MultiHeadLatentAttention().to(DEVICE)
print(mla)
print()
for k, v in mla.kv_cache_size_bytes(SEQ_LEN).items():
    print(f"  {k}: {v}")

MultiHeadLatentAttention(
  (W_DKV): Linear(in_features=8, out_features=4, bias=False)
  (W_UK): Linear(in_features=4, out_features=8, bias=False)
  (W_UV): Linear(in_features=4, out_features=8, bias=False)
  (W_DQ): Linear(in_features=8, out_features=4, bias=False)
  (W_UQ): Linear(in_features=4, out_features=8, bias=False)
  (W_QR): Linear(in_features=4, out_features=4, bias=False)
  (W_KR): Linear(in_features=8, out_features=2, bias=False)
  (W_O): Linear(in_features=8, out_features=8, bias=False)
  (drop): Dropout(p=0.1, inplace=False)
)

  MHA  K+V per token: 2×6 + 2×6 = 24 values
  MLA  c_kv + K_rope: 4 + 2 = 6 values  (K_rope shared, not ×2)
  bytes MHA: 384
  bytes MLA: 96
  compression_ratio: 4.0


## MLA demo

In [111]:
mla.eval()
B = 1

h = torch.randn(B, SEQ_LEN, D_MODEL, device=DEVICE)
print(f"h (input) : {tuple(h.shape)}  (B, T, d_model)")

with torch.no_grad():
    c_kv = mla.W_DKV(h)
    c_q  = mla.W_DQ(h)
    print(f"\n── Step 1 · Compress ──────────────────────────────────────────")
    print(f"c_kv = h @ W_DKV           : {tuple(c_kv.shape)}")
    print(f"c_q  = h @ W_DQ            : {tuple(c_q.shape)}")

    def to_heads(x, d_h):
        return x.view(B, SEQ_LEN, N_HEADS, d_h).transpose(1, 2)

    K_nope = to_heads(mla.W_UK(c_kv), D_HEAD_NOPE)
    V      = to_heads(mla.W_UV(c_kv), D_HEAD_NOPE)
    Q_nope = to_heads(mla.W_UQ(c_q),  D_HEAD_NOPE)
    print(f"\n── Step 2 · NoPE up-projection (per-head) ─────────────────────")
    print(f"K_nope                     : {tuple(K_nope.shape)}  (B, H, T, d_head_nope)")
    print(f"V                          : {tuple(V.shape)}")
    print(f"Q_nope                     : {tuple(Q_nope.shape)}")

    Q_rope = apply_rope(to_heads(mla.W_QR(c_q), D_HEAD_ROPE), freqs_cis)

    K_rope_shared = mla.W_KR(h).unsqueeze(1)                  # (B, 1, T, 2)
    K_rope_shared = apply_rope(K_rope_shared, freqs_cis)       # rotate
    K_rope        = K_rope_shared.expand(-1, N_HEADS, -1, -1) # (B, H, T, 2)

    print(f"\n── Step 3 · Decoupled RoPE ─────────────────────────────────────")
    print(f"Q_rope  [per-head]         : {tuple(Q_rope.shape)}   (B, H, T, d_head_rope)")
    print(f"K_rope  [after W_KR]       : {tuple(mla.W_KR(h).shape)}  → unsqueeze → {tuple(K_rope_shared.shape)}")
    print(f"K_rope  [after broadcast]  : {tuple(K_rope.shape)}   same 2-dim vector in every head")

    Q = torch.cat([Q_nope, Q_rope], dim=-1)
    K = torch.cat([K_nope, K_rope], dim=-1)
    print(f"\n── Step 4 · Concat ─────────────────────────────────────────────")
    print(f"Q = [Q_nope ; Q_rope]      : {tuple(Q.shape)}  (B, H, T, {D_HEAD_NOPE}+{D_HEAD_ROPE})")
    print(f"K = [K_nope ; K_rope]      : {tuple(K.shape)}")

    attn = F.softmax(
        (Q @ K.transpose(-2, -1)) * D_HEAD**-0.5
        .masked_fill(~torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, device=DEVICE, dtype=torch.bool)), float("-inf")),
        dim=-1,
    ) if False else None  # just show shapes
    print(f"\n── Step 5 · Attention ──────────────────────────────────────────")
    print(f"attn scores shape          : (B={B}, H={N_HEADS}, T={SEQ_LEN}, T={SEQ_LEN})")
    out = mla(h, freqs_cis)
    print(f"final output               : {tuple(out.shape)}  ")

    print(f"\n── KV-cache per token ──────────────────────────────────────────")
    print(f"  MHA: K (H×d_head) + V (H×d_head) = {N_HEADS}×{D_HEAD} + {N_HEADS}×{D_HEAD} = {2*N_HEADS*D_HEAD} values")
    print(f"  MLA: c_kv ({D_C_KV}) + K_rope ({D_HEAD_ROPE}, HEAD-SHARED) = {D_C_KV + D_HEAD_ROPE} values")
    print(f"       K_rope is NOT multiplied by n_heads={N_HEADS} — it is broadcast")

h (input) : (1, 8, 8)  (B, T, d_model)

── Step 1 · Compress ──────────────────────────────────────────
c_kv = h @ W_DKV           : (1, 8, 4)
c_q  = h @ W_DQ            : (1, 8, 4)

── Step 2 · NoPE up-projection (per-head) ─────────────────────
K_nope                     : (1, 2, 8, 4)  (B, H, T, d_head_nope)
V                          : (1, 2, 8, 4)
Q_nope                     : (1, 2, 8, 4)

── Step 3 · Decoupled RoPE ─────────────────────────────────────
Q_rope  [per-head]         : (1, 2, 8, 2)   (B, H, T, d_head_rope)
K_rope  [after W_KR]       : (1, 8, 2)  → unsqueeze → (1, 1, 8, 2)
K_rope  [after broadcast]  : (1, 2, 8, 2)   same 2-dim vector in every head

── Step 4 · Concat ─────────────────────────────────────────────
Q = [Q_nope ; Q_rope]      : (1, 2, 8, 6)  (B, H, T, 4+2)
K = [K_nope ; K_rope]      : (1, 2, 8, 6)

── Step 5 · Attention ──────────────────────────────────────────
attn scores shape          : (B=1, H=2, T=8, T=8)
final output               : (1, 8, 8)  

── 

## The Absorption Trick (inference optimisation)

During inference the NoPE content score is:

$$Q_{nope} \cdot K_{nope}^T = (c_Q \cdot W_{UQ}) \cdot (W_{UK} \cdot c_{KV})^T = c_Q \cdot \underbrace{(W_{UQ} \cdot W_{UK}^T)}_{W_{abs}} \cdot c_{KV}^T$$

Pre-compute $W_{abs}$ once → multiply the **low-rank** $c_Q$ and $c_{KV}$ directly. No need to ever materialise the full-dim $K_{nope}$.

In [112]:
# Weight matrices (rows = output, cols = input for nn.Linear, so .weight is (out, in))
# We need them as (in, out) for the math below
W_UQ = mla.W_UQ.weight.T   # (d_c_q,   n_heads*d_head_nope) = (4, 8)
W_UK = mla.W_UK.weight.T   # (d_c_kv,  n_heads*d_head_nope) = (4, 8)

print(f"W_UQ shape : {tuple(W_UQ.shape)}  (d_c_q  × n_heads*d_head_nope)")
print(f"W_UK shape : {tuple(W_UK.shape)}  (d_c_kv × n_heads*d_head_nope)")

# Pre-compute absorbed weight: (d_c_q, d_c_kv) — done ONCE before generation starts
W_abs = W_UQ @ W_UK.T        # (4, 4)
print(f"\nW_abs = W_UQ @ W_UK.T : {tuple(W_abs.shape)}  — precomputed once, lives on GPU")

# ── Naive route (what happens inside forward()) ───────────────────────────────
with torch.no_grad():
    c_q_demo  = mla.W_DQ(h)                             # (B, T, 4)
    c_kv_demo = mla.W_DKV(h)                            # (B, T, 4)

    Q_nope_naive = mla.W_UQ(c_q_demo)                   # (B, T, 8)
    K_nope_naive = mla.W_UK(c_kv_demo)                  # (B, T, 8)
    scores_naive = Q_nope_naive @ K_nope_naive.transpose(-2, -1)  # (B, T, T)

    # ── Absorbed route (skip materialising K_nope entirely) ──────────────────
    # c_q_demo @ W_abs → (B, T, d_c_kv),  then @ c_kv_demo.T → (B, T, T)
    scores_abs = (c_q_demo @ W_abs) @ c_kv_demo.transpose(-2, -1)  # (B, T, T)

    max_diff = (scores_naive - scores_abs).abs().max().item()
    print(f"\nMax difference between naive and absorbed scores: {max_diff:.2e}  (should be ~0)")

    print(f"\nMemory not allocated for K_nope during absorbed inference:")
    k_nope_bytes = B * SEQ_LEN * N_HEADS * D_HEAD_NOPE * 4   # float32
    print(f"  Saved {k_nope_bytes} bytes per forward pass (scales with batch×seq×heads)")

W_UQ shape : (4, 8)  (d_c_q  × n_heads*d_head_nope)
W_UK shape : (4, 8)  (d_c_kv × n_heads*d_head_nope)

W_abs = W_UQ @ W_UK.T : (4, 4)  — precomputed once, lives on GPU

Max difference between naive and absorbed scores: 1.19e-07  (should be ~0)

Memory not allocated for K_nope during absorbed inference:
  Saved 256 bytes per forward pass (scales with batch×seq×heads)


## ShallowSeekMoE

In [113]:
class Expert(nn.Module):
    """A standard feed-forward expert network."""
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False) 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

In [114]:
class ShallowSeekMoE(nn.Module):
    def __init__(self, 
                 d_model: int = D_MODEL, 
                 d_ff_shared: int = 16,       # Shared experts stay relatively large
                 d_ff_routed: int = 2,        # Routed experts are drastically sliced down!
                 n_shared_experts: int = 2,
                 n_routed_experts: int = 32,  # Many more experts, but each is tiny
                 top_k: int = 4,              # Route to top-k experts to recombine concepts
                 bias_update_rate: float = 0.01):
        super().__init__()
        self.d_model = d_model
        self.n_shared_experts = n_shared_experts
        self.n_routed_experts = n_routed_experts
        self.top_k = top_k
        self.bias_update_rate = bias_update_rate

        # 1. Shared Experts (Large capacity, general knowledge)
        self.shared_experts = nn.ModuleList([Expert(d_model, d_ff_shared) for _ in range(n_shared_experts)])

        # 2. Routed Experts (Tiny capacity, hyper-specialized)
        self.routed_experts = nn.ModuleList([Expert(d_model, d_ff_routed) for _ in range(n_routed_experts)])

        # 3. Router Gate Projection
        self.gate = nn.Linear(d_model, n_routed_experts, bias=False)

        # 4. Aux-Loss Free Balancing Biases
        self.register_buffer("expert_biases", torch.zeros(n_routed_experts))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        tokens_flat = x.view(-1, C) 
        num_tokens = tokens_flat.size(0)

        # ─── PATHWAY 1: SHARED EXPERTS ────────────────────────────────────────
        shared_output = torch.zeros_like(tokens_flat)
        for expert in self.shared_experts:
            shared_output += expert(tokens_flat)

        # ─── PATHWAY 2: ROUTED EXPERTS (Fine-Grained) ─────────────────────────
        logits = self.gate(tokens_flat)
        balanced_logits = logits + self.expert_biases.unsqueeze(0)

        scores = F.softmax(balanced_logits, dim=-1)
        topk_weights, topk_indices = torch.topk(scores, self.top_k, dim=-1)
        topk_weights = topk_weights / topk_weights.sum(dim=-1, keepdim=True)

        routed_output = torch.zeros_like(tokens_flat)
        actual_counts = torch.zeros(self.n_routed_experts, device=x.device)

        for i, expert in enumerate(self.routed_experts):
            mask = (topk_indices == i)
            if mask.any():
                token_indices, k_indices = torch.where(mask)
                weights = topk_weights[token_indices, k_indices].unsqueeze(-1)
                
                # Executing the tiny d_ff_routed expert
                expert_out = expert(tokens_flat[token_indices])
                routed_output[token_indices] += weights * expert_out
                
                actual_counts[i] += token_indices.size(0)

        # ─── STEP 3: BIAS UPDATE ──────────────────────────────────────────────
        if self.training:
            target_count = (num_tokens * self.top_k) / self.n_routed_experts
            error = target_count - actual_counts
            self.expert_biases += self.bias_update_rate * error

        final_output = shared_output + routed_output
        return final_output.view(B, T, C)

## Auxiliary Loss Free Load Balancing with finegrained experts Demo

In [115]:
moe_layer = ShallowSeekMoE().to(DEVICE)
mock_input = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=DEVICE)
output = moe_layer(mock_input)
    
print("─── ShallowSeekMoE Fine-Grained Blueprint ───")
print(f"Shared Expert Size   : {moe_layer.shared_experts[0].w1.weight.shape[::-1]} (d_model -> d_ff_shared)")
print(f"Routed Expert Size   : {moe_layer.routed_experts[0].w1.weight.shape[::-1]} (d_model -> d_ff_routed)")
print(f"Active Routed FLOPs  : top_k({moe_layer.top_k}) * d_ff_routed({moe_layer.routed_experts[0].w1.out_features}) = {moe_layer.top_k * moe_layer.routed_experts[0].w1.out_features} active hidden dimensions")

─── ShallowSeekMoE Fine-Grained Blueprint ───
Shared Expert Size   : torch.Size([8, 16]) (d_model -> d_ff_shared)
Routed Expert Size   : torch.Size([8, 2]) (d_model -> d_ff_routed)
Active Routed FLOPs  : top_k(4) * d_ff_routed(2) = 8 active hidden dimensions


## Transformer Block

In [116]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-8):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return x * rms * self.weight

In [117]:
class TransformerBlock(nn.Module):
    """MLA attention + MoE feed-forward, both with pre-RMSNorm + residual."""
    def __init__(self,
                 d_model     = D_MODEL,
                 n_heads     = N_HEADS,
                 d_head_nope = D_HEAD_NOPE,
                 d_head_rope = D_HEAD_ROPE,
                 d_c_kv      = D_C_KV,
                 d_c_q       = D_C_Q,
                 dropout     = DROPOUT):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = MultiHeadLatentAttention(d_model=d_model, n_heads=n_heads, d_head_nope=d_head_nope, 
                                              d_head_rope=d_head_rope, d_c_kv=d_c_kv, d_c_q=d_c_q, dropout=dropout)
        self.norm2 = RMSNorm(d_model)
        self.moe   = ShallowSeekMoE(d_model=d_model)

    def forward(self, x: torch.Tensor, freqs_cis: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), freqs_cis)   # MLA
        x = x + self.moe(self.norm2(x))               # MoE
        return x

## Transformer Block Demo

In [118]:
tf = TransformerBlock().to(DEVICE)
x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=DEVICE)
out = tf(x, freqs_cis)
out[0][0]

tensor([-0.4656, -1.2326, -2.2552,  0.7323,  1.0755,  1.1592,  0.2117,  0.1387],
       device='cuda:0', grad_fn=<SelectBackward0>)

## MultiToken Prediction

In [119]:
class MTPModule(nn.Module):
    """
    One MTP depth-k head.

    Fuses h_prev (from the previous head / main model) with the teacher-forced
    embedding of the k-th future token, then runs a fresh TransformerBlock.
    Embedding table and LM-head weights are SHARED with the main model (tied).
    """
    def __init__(self, d_model: int, shared_embed: nn.Embedding, shared_lm_head: nn.Linear):
        super().__init__()
        self.norm_h   = RMSNorm(d_model)
        self.norm_e   = RMSNorm(d_model)
        self.proj     = nn.Linear(2 * d_model, d_model, bias=False)   # 2D → D
        self.block    = TransformerBlock(d_model=d_model)
        self.norm_out = RMSNorm(d_model)
        # references only — not re-registered as parameters
        self.shared_embed   = shared_embed
        self.shared_lm_head = shared_lm_head

    def forward(self,
                h_prev:      torch.Tensor,   # (B, T_eff, D)  — from prev head
                token_ids_k: torch.Tensor,   # (B, T_eff)     — teacher-forced offset k+1
                freqs_cis:   torch.Tensor,
               ) -> tuple[torch.Tensor, torch.Tensor]:

        e_k    = self.shared_embed(token_ids_k)                              # (B, T_eff, D)
        fused  = torch.cat([self.norm_h(h_prev), self.norm_e(e_k)], dim=-1) # (B, T_eff, 2D)
        h_in   = self.proj(fused)                                             # (B, T_eff, D)
        h_out  = self.block(h_in, freqs_cis)                                 # (B, T_eff, D)
        logits = self.shared_lm_head(self.norm_out(h_out))                   # (B, T_eff, V)
        return h_out, logits

## MTP Demo

In [120]:
print("--- Initializing Shared Main Model Weights ---")
shared_embedding = nn.Embedding(VOCAB_SIZE, D_MODEL)
shared_lm_head = nn.Linear(D_MODEL, VOCAB_SIZE, bias=False)
    
print("--- Initializing MTP Module (Depth k) ---")
mtp_head = MTPModule(d_model=D_MODEL, shared_embed=shared_embedding, shared_lm_head=shared_lm_head)

# 1. h_prev: The hidden state from the main model (or previous MTP head)
h_prev = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL)
    
# 2. token_ids_k: The target tokens shifted by k (Teacher Forced)
token_ids_k = torch.randint(0, VOCAB_SIZE, (BATCH_SIZE, SEQ_LEN))
    
# 3. freqs_cis: Dummy complex rotations for RoPE
# Creating a random complex64 tensor of shape (SEQ_LEN, D_ROPE // 2)
angles = torch.randn(SEQ_LEN, D_HEAD_ROPE // 2)
freqs_cis = torch.polar(torch.ones_like(angles), angles)

print("\n--- Running Forward Pass ---")
h_out, logits = mtp_head(h_prev, token_ids_k, freqs_cis)

# --- Print Validation Results ---
print("\n--- Output Shapes ---")
print(f"h_prev input shape:      {list(h_prev.shape)}    <- (B, T, D_MODEL)")
print(f"token_ids_k shape:       {list(token_ids_k.shape)}          <- (B, T)")
print(f"h_out shape:             {list(h_out.shape)}    <- (B, T, D_MODEL)  | Ready for next MTP head!")
print(f"logits shape:            {list(logits.shape)} <- (B, T, VOCAB_SIZE) | Ready for CrossEntropyLoss!")

--- Initializing Shared Main Model Weights ---
--- Initializing MTP Module (Depth k) ---

--- Running Forward Pass ---

--- Output Shapes ---
h_prev input shape:      [64, 8, 8]    <- (B, T, D_MODEL)
token_ids_k shape:       [64, 8]          <- (B, T)
h_out shape:             [64, 8, 8]    <- (B, T, D_MODEL)  | Ready for next MTP head!
logits shape:            [64, 8, 30000] <- (B, T, VOCAB_SIZE) | Ready for CrossEntropyLoss!


## Complete ShallowSeek

In [121]:
class ShallowSeek(nn.Module):
    """
    embed → TransformerBlock (MLA + MoE) → LM head        ← main model
    └── N_MTP sequential MTP heads                         ← auxiliary training heads
    """
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_mtp=N_MTP):
        super().__init__()
        self.n_mtp = n_mtp

        self.embed    = nn.Embedding(vocab_size, d_model)
        self.block    = TransformerBlock(d_model=d_model)
        self.norm_out = RMSNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight              # weight tying

        self.mtp_modules = nn.ModuleList([MTPModule(d_model, self.embed, self.lm_head) for _ in range(n_mtp)])

    def forward(self, x: torch.Tensor, targets: torch.Tensor = None):
        """
        x       : (B, T)
        targets : (B, T)  — x shifted left by 1  (from dataset)
        """
        B, T = x.shape

        h           = self.embed(x)                           # (B, T, D)
        h           = self.block(h, freqs_cis)                # (B, T, D)  apply_rope slices internally
        main_logits = self.lm_head(self.norm_out(h))          # (B, T, V)

        if targets is None:
            return main_logits, None

        # Shrink to positions that have room for all n_mtp future targets
        # k=n_mtp-1 needs target at t + n_mtp + 1  →  max index = T_eff + n_mtp + 1 - 1 = T
        T_eff = T - self.n_mtp - 1   # e.g. 8 - 3 - 1 = 4

        main_loss = F.cross_entropy(main_logits[:, :T_eff].reshape(-1, main_logits.size(-1)), targets[:, :T_eff].reshape(-1))

        # ── MTP chain ─────────────────────────────────────────────────────────
        # k=0: token_ids_k = x[:,1:T_eff+1],  target = x[:,2:T_eff+2]
        # k=1: token_ids_k = x[:,2:T_eff+2],  target = x[:,3:T_eff+3]
        # k=2: token_ids_k = x[:,3:T_eff+3],  target = x[:,4:T_eff+4]  (= x[:,4:8] ✓)
        h_prev     = h[:, :T_eff]                            # (B, T_eff, D)
        mtp_losses = []

        for k, module in enumerate(self.mtp_modules):
            token_ids_k = x[:,   k + 1 : T_eff + k + 1]    # (B, T_eff)  embed input
            target_k    = x[:,   k + 2 : T_eff + k + 2]    # (B, T_eff)  prediction target

            h_prev, logits_k = module(h_prev, token_ids_k, freqs_cis)

            mtp_losses.append(F.cross_entropy(
                logits_k.reshape(-1, logits_k.size(-1)),
                target_k.reshape(-1)
            ))

        mtp_loss   = sum(mtp_losses) / self.n_mtp
        total_loss = main_loss + MTP_LAMBDA * mtp_loss
        return main_logits, total_loss

## ShallowSeek Demo

In [122]:
model = ShallowSeek().to(DEVICE)
print(model)
print(f"\nTotal params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

x_demo, y_demo = next(iter(dataloader))
x_demo, y_demo = x_demo.to(DEVICE), y_demo.to(DEVICE)

model.train()
logits, loss = model(x_demo, y_demo)
print(f"\nlogits shape : {tuple(logits.shape)}")
print(f"total loss   : {loss.item():.4f}  (main + {MTP_LAMBDA} × avg_mtp)")

ShallowSeek(
  (embed): Embedding(30000, 8)
  (block): TransformerBlock(
    (norm1): RMSNorm()
    (attn): MultiHeadLatentAttention(
      (W_DKV): Linear(in_features=8, out_features=4, bias=False)
      (W_UK): Linear(in_features=4, out_features=8, bias=False)
      (W_UV): Linear(in_features=4, out_features=8, bias=False)
      (W_DQ): Linear(in_features=8, out_features=4, bias=False)
      (W_UQ): Linear(in_features=4, out_features=8, bias=False)
      (W_QR): Linear(in_features=4, out_features=4, bias=False)
      (W_KR): Linear(in_features=8, out_features=2, bias=False)
      (W_O): Linear(in_features=8, out_features=8, bias=False)
      (drop): Dropout(p=0.1, inplace=False)
    )
    (norm2): RMSNorm()
    (moe): ShallowSeekMoE(
      (shared_experts): ModuleList(
        (0-1): 2 x Expert(
          (w1): Linear(in_features=8, out_features=16, bias=False)
          (w2): Linear(in_features=16, out_features=8, bias=False)
          (w3): Linear(in_features=8, out_features=16, bi

## Training Pipeline

In [123]:
model     = ShallowSeek().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for step, (x, y) in enumerate(dataloader):
        x, y = x.to(DEVICE), y.to(DEVICE)

        optimizer.zero_grad()
        _, loss = model(x, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

        if (step + 1) % 200 == 0:
            avg = total_loss / (step + 1)
            print(f"  epoch {epoch} | step {step+1}/{len(dataloader)} | loss {avg:.4f}")

    avg_loss = total_loss / len(dataloader)
    elapsed  = time.time() - t0
    print(f"epoch {epoch} done | avg loss {avg_loss:.4f} | {elapsed:.1f}s")

    ckpt_path = SAVE_DIR / f"shallowseek_epoch{epoch}.pt"
    torch.save({
        "epoch":      epoch,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "loss":        avg_loss,
    }, ckpt_path)

print("training complete.")

  epoch 1 | step 200/2134 | loss 17.9087
  epoch 1 | step 400/2134 | loss 17.2666
  epoch 1 | step 600/2134 | loss 16.5247
  epoch 1 | step 800/2134 | loss 15.7762
  epoch 1 | step 1000/2134 | loss 15.1151
  epoch 1 | step 1200/2134 | loss 14.5627
  epoch 1 | step 1400/2134 | loss 14.0945
  epoch 1 | step 1600/2134 | loss 13.6931
  epoch 1 | step 1800/2134 | loss 13.3439
  epoch 1 | step 2000/2134 | loss 13.0306
epoch 1 done | avg loss 12.8401 | 331.3s
training complete.


In [124]:
def load_checkpoint(path: str) -> ShallowSeek:
    ckpt  = torch.load(path, map_location=DEVICE)
    model = ShallowSeek().to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    print(f"loaded epoch {ckpt['epoch']}  |  loss {ckpt['loss']:.4f}")
    return model

## Inference

In [125]:
@torch.no_grad()
def generate_text(model: ShallowSeek,
                  prompt: str,
                  max_new_tokens: int = 50,
                  temperature: float = 1.0,
                  top_k: int = 50) -> str:

    model.eval()
    prompt_ids = tokenizer.encode(prompt).ids
    ids = torch.tensor(prompt_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)  # (1, T)

    for _ in range(max_new_tokens):
        ids_ctx       = ids[:, -SEQ_LEN:]
        logits, _     = model(ids_ctx)
        logits        = logits[:, -1, :] / temperature          # (1, V)

        if top_k > 0:
            topk_vals, _ = torch.topk(logits, top_k)
            logits        = logits.masked_fill(logits < topk_vals[:, -1:], float("-inf"))
            probs         = F.softmax(logits, dim=-1)
            next_id       = torch.multinomial(probs, num_samples=1)
        else:
            next_id = logits.argmax(dim=-1, keepdim=True)

        ids = torch.cat([ids, next_id], dim=1)

    generated_ids = ids[0, len(prompt_ids):].tolist()          # only the new tokens
    return tokenizer.decode(generated_ids)


In [126]:
model = load_checkpoint("models/shallowseek_epoch1.pt")

response = generate_text(
    model,
    prompt        = "The world is",
    max_new_tokens = 30,
    temperature   = 0.8,
    top_k         = 40,
)
print(response)

loaded epoch 1  |  loss 12.8401
 of it other, as,, from the the upon and upon have have and, of,, and that other the that and and have to,
